# Indirect combustion noise: entropy generation at a flame, conversion at a nozzle

When a flame's heat release fluctuates it does two things: it radiates sound directly, **and**
it leaves a hot/cold *entropy spot* in the burnt gas. That spot is silent while it drifts with
the mean flow, but the moment it is accelerated through a downstream nozzle it radiates sound —
**indirect (entropy) combustion noise**. This notebook walks the whole chain on the Nefes
perturbation network:

1. **Generation** — force the inlet acoustically; the flame's unsteady heat release (an
   $n$–$\tau$ flame transfer function) creates an entropy wave $h$ across the flame. There is no
   entropy upstream, so the flame is unambiguously the source.
2. **Transport** — the entropy spot convects down the hot duct at the mean speed,
   $h(\text{nozzle}) = h(\text{flame})\,e^{-i\omega L/u}$, magnitude conserved.
3. **Conversion** — at the compact choked nozzle the mean-flow acceleration turns the entropy
   spot into a reflected acoustic wave (Marble–Candel): $g = R\,f + R_s\,h$. The $R_s h$ term is
   the indirect noise.

The key modelling switch is **`isentropic=False`** in the forced response: it keeps the convected
entropy characteristic. The default isentropic acoustic analysis pins $h=0$ away from the flame
and therefore *misses indirect noise entirely* — we show that at the end.

In [ ]:
import warnings

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nefes
from nefes.elements import catalog as cat
from nefes.elements import n_tau_flame
from nefes.perturbation import PerturbationBC
from nefes.plotting import use_nefes_theme

use_nefes_theme()
R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
GM1 = GAMMA - 1.0

## 1. A forced flame–nozzle rig

Cold air is metered in (a `mass_flow_inlet`, driven by a unit acoustic excitation), burns at a
compact `heat_release_flame` carrying an $n$–$\tau$ transfer function, and the hot gas exhausts
through a compact `choked_nozzle_outlet`. The choke fixes the approach Mach by its area ratio, so
the hot section runs at a healthy $M$ where entropy noise is appreciable.

![Network topology](entropy_noise_topology.png)

The flame + choke do not converge from the default cold guess (the choke's critical-flux ↔
post-flame stagnation coupling is far away), so we **ramp the heat release** and reuse the
previous state as the guess.

In [ ]:
MDOT, AREA, A_STAR, L1, L2 = 0.4, 0.02, 0.012, 0.5, 0.5
E_COLD, E_FLAME_OUT, E_NOZZLE = 1, 2, 3   # edge ids: flame approach / flame exit / nozzle inlet


def build_rig(n=1.0, tau=2.0e-3, dT=600.0):
    "Forced inlet -> cold duct -> n-tau flame -> hot duct -> choked nozzle (heat-release continuation)."
    ds = n_tau_flame(n, tau, ref_edge=E_COLD)

    def mk(Qdot):
        return nefes.Network(
            nefes.perfect_gas(R, GAMMA),
            nodes=[
                cat.mass_flow_inlet(MDOT, 300.0, perturbation_bc=PerturbationBC.anechoic(driven=("acoustic",))),
                cat.duct(L1, name="cold duct"),
                cat.heat_release_flame(Qdot, name="flame", dynamic_source=ds),
                cat.duct(L2, name="hot duct"),
                cat.choked_nozzle_outlet(A_STAR, name="nozzle"),
            ],
            edges=[(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA)],
        )

    # Ramp the heat release, warm-starting each stage from the previous converged state.
    net, sol = None, None
    for ddT in (1.0, 150.0, 300.0, 450.0, dT):
        net = mk(MDOT * CP * ddT)
        sol = net.solve(x0=sol.x if sol else None)
        assert sol.converged, f"mean solve failed at dT={ddT}"
    return net, sol


net, sol = build_rig()
for lbl, e in [("cold (inlet)", 0), ("flame approach", E_COLD), ("flame exit (hot)", E_FLAME_OUT), ("nozzle", E_NOZZLE)]:
    ed = sol.edge(e)
    print(f"{lbl:<16} M={ed['M']:.3f}  T={ed['T']:7.1f} K  u={ed['u']:7.1f} m/s")
print(f"\nflame raises T from {sol.edge(E_COLD)['T']:.0f} to {sol.edge(E_FLAME_OUT)['T']:.0f} K; "
      f"nozzle approach M = {sol.edge(E_NOZZLE)['M']:.3f} (subsonic).")

## 2. Generation: the flame makes an entropy wave from upstream sound

We force a unit downstream acoustic wave at the inlet and solve the forced perturbation field
over a frequency sweep with `isentropic=False`. Reading the entropy characteristic $|h|$ at each
station shows it is **zero upstream of the flame** and **finite downstream** — created by the
unsteady heat release.

In [ ]:
FREQS = np.linspace(20.0, 800.0, 120)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fr = sol.forced_response(FREQS, isentropic=False)

# |h| at each station for a representative frequency
f0 = 300.0
i0 = int(np.argmin(np.abs(FREQS - f0)))
stations = [("inlet\n(cold)", 0), ("flame\napproach", E_COLD), ("flame exit\n(hot)", E_FLAME_OUT), ("nozzle\n(hot)", E_NOZZLE)]
h_station = [abs(fr.waves(e)[i0, 2]) for _, e in stations]
fig = go.Figure(go.Bar(x=[s for s, _ in stations], y=h_station,
                       marker_color=["#1f77b4", "#1f77b4", "#d62728", "#d62728"]))
fig.update_layout(title=f"Entropy-wave amplitude |h| along the rig at {FREQS[i0]:.0f} Hz",
                  yaxis_title="|h|  (entropy characteristic)",
                  annotations=[dict(x=1.5, y=max(h_station), text="flame generates entropy →",
                                    showarrow=False, yshift=18, font=dict(color="#d62728"))])
fig.show()
print(f"upstream of flame |h| = {h_station[1]:.2e}   downstream |h| = {h_station[2]:.2e}")

The entropy spot is generated only across the flame. Its **generation transfer function**
$h_\text{hot}/f_\text{inlet}$ over frequency carries the flame's $n$–$\tau$ fingerprint (the
$e^{-i\omega\tau}$ lag beating against the duct acoustics):

In [ ]:
h_hot = fr.waves(E_FLAME_OUT)[:, 2]   # entropy just downstream of the flame (forcing f_inlet = 1)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("magnitude", "phase"))
fig.add_scatter(x=FREQS, y=np.abs(h_hot), mode="lines", name="|h_hot|", row=1, col=1)
fig.add_scatter(x=FREQS, y=np.degrees(np.angle(h_hot)), mode="lines", name="phase", row=2, col=1)
fig.update_yaxes(title_text="|h_hot / f_in|", row=1, col=1)
fig.update_yaxes(title_text="deg", row=2, col=1)
fig.update_xaxes(title_text="frequency [Hz]", row=2, col=1)
fig.update_layout(title="Entropy generation transfer function (flame exit)", showlegend=False)
fig.show()

## 3. Transport: the entropy spot convects to the nozzle

Between flame and nozzle the entropy wave is just convected by the mean flow — no generation, no
loss. So $h(\text{nozzle}) = h(\text{flame})\,e^{-i\omega L_2/u_\text{hot}}$ exactly, with the
magnitude unchanged. We overlay the measured nozzle entropy on this analytic convective lag.

In [ ]:
u_hot = sol.edge(E_FLAME_OUT)["u"]
h_nz = fr.waves(E_NOZZLE)[:, 2]
omega = 2.0 * np.pi * FREQS
h_pred = h_hot * np.exp(-1j * omega * L2 / u_hot)
print(f"max |h_nozzle - h_flame*exp(-i w L/u)| / max|h| = {np.max(np.abs(h_nz - h_pred))/np.max(np.abs(h_nz)):.2e}")
print(f"magnitude conserved:  max| |h_nz| - |h_flame| | = {np.max(np.abs(np.abs(h_nz)-np.abs(h_hot))):.2e}")
fig = go.Figure()
fig.add_scatter(x=FREQS, y=np.degrees(np.angle(h_nz)), mode="lines", name="measured nozzle phase")
fig.add_scatter(x=FREQS, y=np.degrees(np.angle(h_pred)), mode="lines", line=dict(dash="dash"),
                name="convective lag  exp(-i w L/u)")
fig.update_layout(title="Entropy phase at the nozzle: pure convective transport",
                  xaxis_title="frequency [Hz]", yaxis_title="phase [deg]")
fig.show()

## 4. Conversion: the nozzle turns the entropy spot into sound

At the compact choked nozzle the inherited closure is Marble–Candel,
$g = R\,f + R_s\,h$. The first term is the **direct** reflection of the acoustic wave; the
second, $R_s h$, is the **indirect** (entropy) noise. We split the reflected wave into the two
and compare their magnitudes across frequency — the indirect part is a large fraction of the
total here.

In [ ]:
ed = sol.edge(E_NOZZLE)
rho_n, c_n, M_n = ed["rho"], ed["c"], ed["M"]
R_mc = (2.0 - GM1 * M_n) / (2.0 + GM1 * M_n)
Rs_mc = (c_n / rho_n) * M_n / (2.0 + GM1 * M_n)
f_nz, g_nz = fr.waves(E_NOZZLE)[:, 0], fr.waves(E_NOZZLE)[:, 1]
direct, indirect = R_mc * f_nz, Rs_mc * h_nz
print(f"nozzle: M={M_n:.3f}  R={R_mc:.4f}  R_s={Rs_mc:.1f}")
print(f"closure g = R f + R_s h holds to {np.max(np.abs(g_nz-(direct+indirect)))/np.max(np.abs(g_nz)):.1e}")
fig = go.Figure()
fig.add_scatter(x=FREQS, y=np.abs(g_nz), mode="lines", name="|g| total reflected", line=dict(width=3))
fig.add_scatter(x=FREQS, y=np.abs(direct), mode="lines", name="|R f|  direct")
fig.add_scatter(x=FREQS, y=np.abs(indirect), mode="lines", name="|R_s h|  indirect (entropy noise)",
                line=dict(color="#d62728"))
fig.update_layout(title="Reflected wave at the choked nozzle: direct vs indirect noise",
                  xaxis_title="frequency [Hz]", yaxis_title="reflected acoustic amplitude")
fig.show()

## 5. The default isentropic analysis misses indirect noise

The standard isentropic acoustic mode pins $h=0$ everywhere except at the flame itself, so the
entropy spot never reaches the nozzle and never converts. Re-running with `isentropic=True`, the
nozzle entropy collapses to zero and the reflected wave differs from the full answer — the gap
**is** the indirect noise.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fr_isen = sol.forced_response(FREQS, isentropic=True)
g_isen = fr_isen.waves(E_NOZZLE)[:, 1]
print(f"max |h| at nozzle, isentropic=True : {np.max(np.abs(fr_isen.waves(E_NOZZLE)[:,2])):.2e}  (pinned to 0)")
print(f"max |h| at nozzle, isentropic=False: {np.max(np.abs(h_nz)):.2e}")
fig = go.Figure()
fig.add_scatter(x=FREQS, y=np.abs(g_nz), mode="lines", name="isentropic=False (with entropy noise)")
fig.add_scatter(x=FREQS, y=np.abs(g_isen), mode="lines", line=dict(dash="dash"),
                name="isentropic=True (entropy dropped)")
fig.update_layout(title="Reflected wave with vs without the entropy path",
                  xaxis_title="frequency [Hz]", yaxis_title="|g| at the nozzle")
fig.show()

## Summary

* A flame with an unsteady heat-release response **generates an entropy wave** from upstream
  acoustic forcing — finite downstream, zero upstream (§2). The generation transfer function
  carries the $n$–$\tau$ signature.
* The entropy spot **convects** to the nozzle at the mean speed, $h\propto e^{-i\omega L/u}$,
  magnitude conserved (§3).
* The compact choked nozzle **converts** it to sound via the Marble–Candel coupling
  $g = R f + R_s h$; the indirect ($R_s h$) part is a major share of the reflected wave (§4).
* All of this requires retaining the convected entropy characteristic (`isentropic=False`); the
  default isentropic acoustic analysis drops it and misses indirect noise (§5).

This is the regime where the contour eigensolver struggles (long convective entropy delays
overflow at complex $\omega$); the planned Nyquist / real-frequency-sweep stability driver is the
right tool for stability questions here.

## Export for Nemo

The flame–nozzle rig is available as `net` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `net.to_yaml(path)` writes the topology only — then open the file in Nemo.

In [ ]:
import os, tempfile

_out = os.path.join(tempfile.mkdtemp(), "entropy_noise.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use net.to_yaml(_out) for topology only
print("saved case:", _out)